# 面试问题：LoRA 与 QLoRA 的核心机制如何从零实现？

可以直接复述的回答是：LoRA 冻结预训练权重 `W`，只学习低秩增量 `ΔW=(α/r)BA`，因此训练参数和优化器状态显著减少。通常 A 用小随机数初始化、B 初始化为零，使训练起点严格等于基础模型且第一步 B 有非零梯度。QLoRA 进一步把冻结的 W 量化保存，前向时反量化参与矩阵乘法，梯度仍只流向高精度 adapter。量化误差、rank、alpha 和注入层决定效果。下面不用 PEFT 或 Trainer，手写线性 LoRA、逐行 INT4 量化和真实 backward。

## 真实案例：把通用客服分类器适配到退款、物流和技术故障

八条工单由四个脱敏语义特征表示：退款、物流、故障和紧急度。基础权重故意来自类别映射不同的旧域，用于观察少量 adapter 参数如何完成域适配；小样本准确率不能外推线上系统。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量与自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持教学输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(1301)  # 固定 adapter 初始化
label_names = ["退款", "物流", "技术"]  # 定义三个客服意图名称
records = [  # 定义八条带业务语义的工单向量
    ("A-01", "重复扣款要求退款", [1.0, 0.1, 0.1, 0.8], 0),  # 高紧急退款工单
    ("A-02", "商品退货到账", [0.9, 0.2, 0.1, 0.4], 0),  # 普通退款工单
    ("A-03", "退款进度查询", [0.8, 0.2, 0.0, 0.3], 0),  # 退款进度工单
    ("A-04", "包裹三天未更新", [0.1, 1.0, 0.1, 0.7], 1),  # 高紧急物流工单
    ("A-05", "修改配送地址", [0.1, 0.9, 0.1, 0.3], 1),  # 普通物流工单
    ("A-06", "设备无法开机", [0.1, 0.1, 1.0, 0.9], 2),  # 高紧急技术工单
    ("A-07", "应用持续闪退", [0.1, 0.2, 0.9, 0.8], 2),  # 软件故障工单
    ("A-08", "蓝牙连接失败", [0.2, 0.1, 0.8, 0.5], 2),  # 连接故障工单
]  # 结束八条适配样本
x = torch.tensor([row[2] for row in records], dtype=torch.float32)  # 构造八乘四语义特征矩阵
y = torch.tensor([row[3] for row in records], dtype=torch.long)  # 构造三分类标签
base_weight = torch.tensor([[0.0, 1.2, 0.0, 0.1], [0.0, 0.0, 1.2, 0.1], [1.2, 0.0, 0.0, 0.1]], dtype=torch.float32)  # 定义旧域类别循环错位的冻结权重
print("输入预览：id | text | features | label")  # 输出领域适配样本表头
for row in records:  # 逐条展示八条工单
    print(f"{row[0]} | {row[1]:10} | {row[2]} | {label_names[row[3]]}")  # 展示可读文本和数值特征
print("基础权重形状：", tuple(base_weight.shape))  # 展示待适配线性层尺寸

输入预览：id | text | features | label
A-01 | 重复扣款要求退款   | [1.0, 0.1, 0.1, 0.8] | 退款
A-02 | 商品退货到账     | [0.9, 0.2, 0.1, 0.4] | 退款
A-03 | 退款进度查询     | [0.8, 0.2, 0.0, 0.3] | 退款
A-04 | 包裹三天未更新    | [0.1, 1.0, 0.1, 0.7] | 物流
A-05 | 修改配送地址     | [0.1, 0.9, 0.1, 0.3] | 物流
A-06 | 设备无法开机     | [0.1, 0.1, 1.0, 0.9] | 技术
A-07 | 应用持续闪退     | [0.1, 0.2, 0.9, 0.8] | 技术
A-08 | 蓝牙连接失败     | [0.2, 0.1, 0.8, 0.5] | 技术
基础权重形状： (3, 4)


## Baseline / 基线：冻结旧域权重直接推理

基础模型把三个核心特征循环映射到了错误标签。它不更新任何参数，作为 LoRA 和 QLoRA 的同数据起点。

In [2]:
def stable_cross_entropy(logits, labels):  # 手写数值稳定多分类交叉熵
    shifted = logits - logits.max(dim=1, keepdim=True).values  # 减去每行最大 logit 防止指数溢出
    log_normalizer = torch.log(torch.exp(shifted).sum(dim=1))  # 计算每条样本对数归一化项
    selected = shifted[torch.arange(len(labels)), labels]  # 选取真实类别缩放后 logit
    return (log_normalizer - selected).mean()  # 返回平均负对数似然
with torch.no_grad():  # 关闭冻结基础模型的梯度记录
    baseline_logits = x @ base_weight.T  # 使用旧域权重执行真实前向
    baseline_prediction = baseline_logits.argmax(dim=1)  # 获取基础模型类别预测
baseline_accuracy = float((baseline_prediction == y).float().mean())  # 计算同八样本基线准确率
print("id | label | baseline_pred | logits")  # 输出基础模型逐样本结果表头
for index, row in enumerate(records):  # 遍历八条工单
    print(f"{row[0]} | {label_names[int(y[index])]} | {label_names[int(baseline_prediction[index])]} | {[round(value, 3) for value in baseline_logits[index].tolist()]}")  # 展示类别错位细节
print(f"冻结基础模型 accuracy={baseline_accuracy:.1%}")  # 汇总领域错配基线

id | label | baseline_pred | logits
A-01 | 退款 | 技术 | [0.2, 0.2, 1.28]
A-02 | 退款 | 技术 | [0.28, 0.16, 1.12]
A-03 | 退款 | 技术 | [0.27, 0.03, 0.99]
A-04 | 物流 | 退款 | [1.27, 0.19, 0.19]
A-05 | 物流 | 退款 | [1.11, 0.15, 0.15]
A-06 | 技术 | 物流 | [0.21, 1.29, 0.21]
A-07 | 技术 | 物流 | [0.32, 1.16, 0.2]
A-08 | 技术 | 物流 | [0.17, 1.01, 0.29]
冻结基础模型 accuracy=0.0%


## 核心实现：LoRA `W + (α/r)BA`

基础权重注册为 buffer，不参与更新。A 为 `rank×in`，B 为 `out×rank`；训练只手写更新这两个 Parameter。

In [3]:
class LoRALinear(torch.nn.Module):  # 定义不依赖现成 PEFT 的低秩线性层
    def __init__(self, frozen_weight, rank=2, alpha=4.0, zero_both=False):  # 初始化冻结权重与 adapter
        super().__init__()  # 初始化 PyTorch 模块基类
        self.register_buffer("base_weight", frozen_weight.clone())  # 保存不参与梯度的基础权重
        self.rank = rank  # 保存低秩维度供缩放使用
        self.alpha = alpha  # 保存 LoRA 强度超参数
        initial_a = torch.zeros(rank, frozen_weight.shape[1]) if zero_both else torch.randn(rank, frozen_weight.shape[1]) * 0.05  # 正确方案随机初始化 A
        self.adapter_a = torch.nn.Parameter(initial_a)  # 注册输入到低秩空间矩阵
        self.adapter_b = torch.nn.Parameter(torch.zeros(frozen_weight.shape[0], rank))  # 注册零初始化低秩到输出矩阵
    def forward(self, features):  # 定义基础路径和低秩增量前向
        base_output = features @ self.base_weight.T  # 计算冻结基础权重输出
        low_rank_output = (features @ self.adapter_a.T) @ self.adapter_b.T  # 依次经过 A 和 B 而不显式构造大矩阵
        return base_output + self.alpha / self.rank * low_rank_output  # 合并基础输出和缩放增量
def train_adapter(model, steps=180, learning_rate=0.20):  # 手写 adapter 全批量训练循环
    trace = []  # 保存关键步损失、梯度和增量范数
    for step in range(1, steps + 1):  # 执行固定预算领域适配
        logits = model(x)  # 真实执行 LoRA forward
        loss = stable_cross_entropy(logits, y)  # 计算三分类监督损失
        loss.backward()  # 真实执行 backward 得到 A 与 B 梯度
        gradient_a = float(torch.linalg.vector_norm(model.adapter_a.grad))  # 记录 A 梯度范数
        gradient_b = float(torch.linalg.vector_norm(model.adapter_b.grad))  # 记录 B 梯度范数
        with torch.no_grad():  # 关闭手写参数更新的计算图
            model.adapter_a -= learning_rate * model.adapter_a.grad  # 只更新低秩矩阵 A
            model.adapter_b -= learning_rate * model.adapter_b.grad  # 只更新低秩矩阵 B
            model.adapter_a.grad.zero_()  # 清空 A 梯度
            model.adapter_b.grad.zero_()  # 清空 B 梯度
        if step in {1, 2, 20, steps}:  # 保存最能解释训练机制的节点
            delta = model.alpha / model.rank * (model.adapter_b @ model.adapter_a)  # 显式计算当前低秩权重增量供观察
            trace.append((step, float(loss), gradient_a, gradient_b, float(torch.linalg.vector_norm(delta))))  # 保存损失、梯度和增量范数
    return trace  # 返回真实训练轨迹
lora_model = LoRALinear(base_weight)  # 创建 rank 二 LoRA 模型
lora_trace = train_adapter(lora_model)  # 只训练两个 adapter 参数
with torch.no_grad():  # 进入 LoRA 评估阶段
    lora_logits = lora_model(x)  # 计算八条样本适配后 logits
    lora_prediction = lora_logits.argmax(dim=1)  # 获取 LoRA 类别预测
lora_accuracy = float((lora_prediction == y).float().mean())  # 计算 LoRA 训练样本准确率
print("step | loss | grad_A | grad_B | ||delta_W||")  # 输出 LoRA 中间轨迹表头
for item in lora_trace:  # 遍历四个关键训练节点
    print(f"{item[0]:4d} | {item[1]:.5f} | {item[2]:.5f} | {item[3]:.5f} | {item[4]:.5f}")  # 展示低秩参数真实更新
print(f"LoRA accuracy={lora_accuracy:.1%}，trainable={sum(parameter.numel() for parameter in lora_model.parameters())}")  # 汇总效果和训练参数量

step | loss | grad_A | grad_B | ||delta_W||
   1 | 1.50407 | 0.00000 | 0.07761 | 0.00264
   2 | 1.50286 | 0.01072 | 0.07758 | 0.00536
  20 | 1.23854 | 0.38893 | 0.39338 | 0.78211
 180 | 0.00417 | 0.00917 | 0.00916 | 12.11673
LoRA accuracy=100.0%，trainable=14


## QLoRA：逐输出通道 INT4 基座加 FP32 Adapter

每行 scale 为 `max(abs(W_row))/7`，整数码限制在 `[-7,7]`。这里只模拟反量化计算，没有调用量化库或真实 INT4 kernel。

In [4]:
def quantize_int4_per_row(weight):  # 手写冻结权重逐行对称 INT4 量化
    scale = weight.abs().amax(dim=1, keepdim=True) / 7.0  # 计算每个输出通道的量化步长
    safe_scale = torch.where(scale > 0.0, scale, torch.ones_like(scale))  # 避免全零行产生除零
    codes = torch.round(weight / safe_scale).clamp(-7, 7).to(torch.int8)  # 生成可观察四位有符号整数码
    return codes, safe_scale  # 返回整数权重和逐行 scale
class QLoRALinear(LoRALinear):  # 定义量化基座上的低秩适配层
    def __init__(self, frozen_weight, rank=2, alpha=4.0):  # 初始化 INT4 buffer 和 FP32 adapter
        super().__init__(frozen_weight, rank, alpha)  # 复用 LoRA adapter 参数初始化
        codes, scale = quantize_int4_per_row(frozen_weight)  # 对基础权重执行逐行量化
        self.register_buffer("base_codes", codes)  # 保存不参与梯度的 INT4 整数码
        self.register_buffer("base_scale", scale)  # 保存反量化逐行 scale
    def forward(self, features):  # 定义量化基座与低秩增量前向
        dequantized_weight = self.base_codes.float() * self.base_scale  # 在教学前向中反量化冻结权重
        base_output = features @ dequantized_weight.T  # 使用量化近似基座计算输出
        low_rank_output = (features @ self.adapter_a.T) @ self.adapter_b.T  # 计算 FP32 LoRA 增量
        return base_output + self.alpha / self.rank * low_rank_output  # 合并量化基座和低秩更新
qlora_model = QLoRALinear(base_weight)  # 创建量化基座适配模型
qlora_trace = train_adapter(qlora_model)  # 真实训练 QLoRA adapter
with torch.no_grad():  # 进入 QLoRA 评估阶段
    qlora_logits = qlora_model(x)  # 计算八条工单量化适配输出
    qlora_prediction = qlora_logits.argmax(dim=1)  # 获取 QLoRA 类别预测
qlora_accuracy = float((qlora_prediction == y).float().mean())  # 计算量化适配准确率
base_fp32_bytes = base_weight.numel() * 4  # 估算基础权重 FP32 存储字节
base_int4_bytes = (base_weight.numel() + 1) // 2 + base_weight.shape[0] * 4  # 估算打包 INT4 码和 FP32 行 scale 字节
adapter_bytes = sum(parameter.numel() * 4 for parameter in qlora_model.parameters())  # 估算 FP32 adapter 字节
print("INT4 codes：", qlora_model.base_codes.tolist())  # 展示真实整数码
print("row scales：", [round(value, 5) for value in qlora_model.base_scale.flatten().tolist()])  # 展示逐通道反量化参数
print(f"QLoRA accuracy={qlora_accuracy:.1%}，base FP32={base_fp32_bytes}B，INT4+scale={base_int4_bytes}B，adapter={adapter_bytes}B")  # 汇总效果和教学存储估算
print("id | label | baseline | LoRA | QLoRA")  # 输出逐样本对照表头
for index, row in enumerate(records):  # 遍历八条领域工单
    print(f"{row[0]} | {label_names[int(y[index])]} | {label_names[int(baseline_prediction[index])]} | {label_names[int(lora_prediction[index])]} | {label_names[int(qlora_prediction[index])]}")  # 展示两种适配结果

INT4 codes： [[0, 7, 0, 1], [0, 0, 7, 1], [7, 0, 0, 1]]
row scales： [0.17143, 0.17143, 0.17143]
QLoRA accuracy=100.0%，base FP32=48B，INT4+scale=18B，adapter=56B
id | label | baseline | LoRA | QLoRA
A-01 | 退款 | 技术 | 退款 | 退款
A-02 | 退款 | 技术 | 退款 | 退款
A-03 | 退款 | 技术 | 退款 | 退款
A-04 | 物流 | 退款 | 物流 | 物流
A-05 | 物流 | 退款 | 物流 | 物流
A-06 | 技术 | 物流 | 技术 | 技术
A-07 | 技术 | 物流 | 技术 | 技术
A-08 | 技术 | 物流 | 技术 | 技术


## 失败案例与修正：A、B 同时初始化为零

若 A 和 B 都为零，`∂L/∂A` 包含 B，`∂L/∂B` 包含 A，第一步两个梯度都会是零，训练无法启动。正确做法是随机 A、零 B：起始输出仍等于基础模型，但 B 第一层梯度非零，更新后 A 也获得梯度。

In [5]:
zero_model = LoRALinear(base_weight, zero_both=True)  # 创建 A 与 B 都为零的错误模型
zero_loss = stable_cross_entropy(zero_model(x), y)  # 计算错误初始化的首步损失
zero_loss.backward()  # 真实执行错误模型首步 backward
zero_grad_a = float(torch.linalg.vector_norm(zero_model.adapter_a.grad))  # 测量错误 A 梯度
zero_grad_b = float(torch.linalg.vector_norm(zero_model.adapter_b.grad))  # 测量错误 B 梯度
fixed_probe = LoRALinear(base_weight, zero_both=False)  # 创建随机 A、零 B 的正确探针
fixed_loss = stable_cross_entropy(fixed_probe(x), y)  # 计算正确初始化首步损失
fixed_loss.backward()  # 真实执行正确探针首步 backward
fixed_grad_a = float(torch.linalg.vector_norm(fixed_probe.adapter_a.grad))  # 测量正确方案首步 A 梯度
fixed_grad_b = float(torch.linalg.vector_norm(fixed_probe.adapter_b.grad))  # 测量正确方案首步 B 梯度
print(f"双零初始化：grad_A={zero_grad_a:.6f}，grad_B={zero_grad_b:.6f}")  # 展示训练完全停滞
print(f"随机A/零B：grad_A={fixed_grad_a:.6f}，grad_B={fixed_grad_b:.6f}")  # 展示保持原输出同时启动 B 更新

双零初始化：grad_A=0.000000，grad_B=0.000000
随机A/零B：grad_A=0.000000，grad_B=0.085343


## 结果解读

LoRA 初始 B 为零，因此第 1 步 A 梯度为零是正常现象；B 更新后低秩路径打开，A 从第 2 步开始学习。QLoRA 的基础权重整数码和 scale 已实际参与 forward，训练参数仍只有 A、B。这里的高准确率来自八条受控样本，不代表真实语言模型能力。

## 生产边界

教学层只有 `3×4` 权重，INT4 也未打包成真实 kernel。生产 QLoRA 还涉及 NF4、double quant、paged optimizer、计算 dtype、梯度 checkpoint 和多卡通信。必须评估 held-out 任务、灾难性遗忘、不同 rank/target modules，并保存基础模型、tokenizer、量化配置和 adapter 版本。

## 最小回归测试

In [6]:
assert len(records) >= 6  # 保证领域适配案例包含多个真实语义样本
assert baseline_accuracy < lora_accuracy and lora_accuracy >= 0.875  # 保证真实 LoRA 更新改善旧域基线
assert qlora_accuracy >= 0.875  # 保证量化基座上的 adapter 保持基本适配效果
assert all(not parameter.requires_grad for name, parameter in lora_model.named_buffers())  # 保证基础权重以冻结 buffer 保存
assert int(qlora_model.base_codes.min()) >= -7 and int(qlora_model.base_codes.max()) <= 7  # 保证基础权重落在对称 INT4 码范围
assert zero_grad_a == 0.0 and zero_grad_b == 0.0  # 保证双零初始化停滞失败真实复现
assert fixed_grad_b > 0.0  # 保证随机 A、零 B 能启动首步学习